# Diagnóstico de cancro da mama — Passive Learning (baseline)

**Inteligência Artificial Centrada no Humano** · Mestrado em Inteligência Artificial e Engenharia de Dados · FCTUC

Treina a CNN com **todo** o conjunto de treino anotado. Os resultados servem de referência para as experiências de Active Learning (`02_active_learning.ipynb`), que usam muito menos imagens anotadas.

> **Estado**
> - ✅ Feito: código completo e testado com uma configuração reduzida; métricas de recall, F1 e falsos negativos.
> - ⬜ Falta: correr o treino completo (ainda não há resultados) e escrever as conclusões.

**Índice**
1. Configuração
2. Dados
3. Treino
4. Avaliação
5. Conclusões

## 1. Configuração

In [ ]:
# Ambiente: monta o Google Drive no Colab; localmente usa a pasta do repositório
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/My Drive/miacd-iach-breast-cancer-diagnosis")  # pasta do repositório no Drive
    %pip install -q modAL-python scikeras
else:
    ROOT = Path.cwd().parent

sys.path.insert(0, str(ROOT))
from src.config import DATA_PATH, RESULTS_PATH

In [ ]:
# Parâmetros
IMG_SIZE = 224          # reduzir (ex.: 128) se faltar memória
EPOCHS = 15

## 2. Dados

In [ ]:
# Carregar, dividir em treino/teste e balancear o treino (o teste fica com as imagens originais)
import numpy as np
from src.data import CLASSES, load_dataset, summarize

x_train, x_test, y_train, y_test = load_dataset(DATA_PATH, size=IMG_SIZE)
summarize(x_train, x_test, y_train, y_test)

## 3. Treino

A arquitetura da CNN está em `src/model.py`. 20% do treino é usado para validação.

In [ ]:
from tensorflow.keras.utils import to_categorical
from src.model import build_cnn

y_train_one_hot = to_categorical(y_train, num_classes=len(CLASSES))

model = build_cnn(size=IMG_SIZE)
history = model.fit(x_train, y_train_one_hot, epochs=EPOCHS, validation_split=0.2, verbose=1)

## 4. Avaliação

O foco é o **recall de maligno** e os **falsos negativos** (maligno classificado como benigno ou normal). As métricas ficam em `results/passive_learning.json` para o notebook 02 as usar como baseline.

In [ ]:
from src.evaluation import compute_metrics, save_json

y_pred = np.argmax(model.predict(x_test, verbose=0), axis=1)
metrics = compute_metrics(y_test, y_pred)
metrics["accuracy_train"] = float(model.evaluate(x_train, y_train_one_hot, verbose=0)[1])
metrics["accuracy_test"] = metrics["accuracy"]

save_json(metrics, RESULTS_PATH / "passive_learning.json")
metrics

In [ ]:
from src.evaluation import plot_confusion_matrix

plot_confusion_matrix(y_test, y_pred, title="Passive Learning")

## 5. Conclusões

*A preencher depois de correr o treino completo.*